In [2]:
 import requests
from typing import List, Optional, Dict, Any

In [3]:
def get_collection_leaderboard(
    api_key: str,
    blockchains: Optional[List[str]] = None,
    period: str = "D7",
    sort: str = "VOLUME",
    direction: str = "DESC",
    limit: int = 10,
    max_records: Optional[int] = None,
) -> List[Dict[str, Any]]:
    """
    Fetches the NFT Collections leaderboard from Rarible, applying optional filters and paging through results.

    Args:
        api_key:       Your Rarible API key.
        blockchains:   Optional list of blockchains to include (e.g. ["ETHEREUM", "POLYGON"]).
        period:        Time period for aggregation. One of MIN5, MIN30, H1, H6, D1, D7, D30, D180, D365, ALL.
                       Default "D7". :contentReference[oaicite:0]{index=0}
        sort:          Field to sort by. One of VOLUME, VOLUME_CHANGE, FLOOR, FLOOR_CHANGE, SALES, SALES_CHANGE, LISTED, OWNERS.
                       Default "VOLUME". :contentReference[oaicite:1]{index=1}
        direction:     Sort direction, "ASC" or "DESC". Default "DESC". :contentReference[oaicite:2]{index=2}
        limit:         Number of records per page. Default 10. :contentReference[oaicite:3]{index=3}
        max_records:   If set, stops once this many total records have been fetched.

    Returns:
        A list of leaderboard entry dicts (each containing `id`, `sales`, `volume`, `floorChangePercent`, etc.).
        See response schema for full fields. :contentReference[oaicite:4]{index=4}
    """
    endpoint = "https://api.rarible.org/v2.0/data/leaderboard/collections"
    headers = {
        "X-API-KEY": api_key,
        "Accept": "application/json",
    }
    params: Dict[str, Any] = {
        "period": period,
        "sort": sort,
        "direction": direction,
    }
    if blockchains:
        params["blockchains"] = blockchains

    all_results: List[Dict[str, Any]] = []
    offset = 0

    while True:
        params.update({
            "limit": limit,
            "offset": offset,
        })

        resp = requests.get(endpoint, headers=headers, params=params)
        resp.raise_for_status()
        data = resp.json()

        # The endpoint returns a "result" array of entries
        results = data.get("result", [])
        all_results.extend(results)

        # If we've reached max_records, trim and return
        if max_records and len(all_results) >= max_records:
            return all_results[:max_records]

        # If fewer results than requested, we're at the end
        if len(results) < limit:
            break

        offset += limit

    return all_results


# Example usage:
if __name__ == "__main__":
    API_KEY = "7162ca80-e2bc-424a-9e3d-57d5be0f9944"
    top_collections = get_collection_leaderboard(
        api_key=API_KEY,
        blockchains=["ETHEREUM", "POLYGON"],
        period="D7",
        sort="VOLUME",
        direction="DESC",
        limit=20,
        max_records=100  # fetch up to 100 entries
    )
    print(f"Fetched {len(top_collections)} collections")
    # Inspect the first entry
    if top_collections:
        import json
        print(json.dumps(top_collections[0], indent=2))


Fetched 0 collections


In [13]:
import requests
from typing import List, Optional

def check_balance(
    api_key: str,
    currency_id: str,
    owner: str
) -> int:
    """
    Fetches the balance of a specific currency for a given owner.

    Args:
        api_key:     Your Rarible API key.
        currency_id: Currency identifier, e.g. "ETHEREUM:0x...".
                     Can also be ERC-1155 style: "ETHEREUM:0x...:{tokenId}".
        owner:       Owner address with chain prefix, e.g. "ETHEREUM:0x...".

    Returns:
        The raw balance (integer). You can divide by the token's decimals yourself
        or inspect the 'decimal' field in the full JSON if you need the human‑readable amount.

    Raises:
        HTTPError: If the request fails (status code ≠ 200).
    """
    endpoint = f"https://api.rarible.org/v0.1/balances/{currency_id}/{owner}"
    headers = {
        "X-API-KEY": api_key,
        "Accept": "application/json",
    }

    resp = requests.get(endpoint, headers=headers)
    resp.raise_for_status()
    data = resp.json()
    # Example response fields: "balance": 123456, "decimal": 123456.789 :contentReference[oaicite:0]{index=0}
    return data


def get_nft_amount(
    api_key: str,
    owner: str,
    blockchains: Optional[List[str]] = None,
) -> int:
    """
    Returns the total number of NFTs owned by a given address.

    Args:
        api_key:     Your Rarible API key.
        owner:       Wallet address with chain prefix, e.g. "ETHEREUM:0x...".
        blockchains: Optional list of chains to filter by (e.g. ["ETHEREUM","POLYGON"]).
                     If omitted, counts NFTs across all supported chains.

    Returns:
        The total count of NFTs owned by `owner`.

    Raises:
        HTTPError: If the request fails (status code ≠ 200).
    """
    endpoint = "https://api.rarible.org/v0.1/items/byOwner"
    headers = {
        "X-API-KEY": api_key,
        "Accept": "application/json",
    }
    params = {
        "owner": owner,
        "size": 1,  # we only need 1 item to get the `total` field
    }
    if blockchains:
        params["blockchains"] = blockchains

    resp = requests.get(endpoint, headers=headers, params=params)
    resp.raise_for_status()
    data = resp.json()
    # Response contains: "total": <int>, "items": [...] :contentReference[oaicite:1]{index=1}
    # return data.get("total", len(data.get("items", [])))
    return data


# Example usage:
if __name__ == "__main__":
    API_KEY = "7162ca80-e2bc-424a-9e3d-57d5be0f9944"
    # 1) Check ETH balance
    bal = check_balance(
        api_key='0x5f71C27669e8Ca80915EdF25C7909e2B1FD1d939',
        currency_id="ETHEREUM:0x0000000000000000000000000000000000000000",
        owner="ETHEREUM:0x4765273c477c2dc484da4f1984639e943adccfeb"
    )
    print(f"Raw balance: {bal}")

    # 2) Count NFTs
    nft_count = get_nft_amount(
        api_key='7162ca80-e2bc-424a-9e3d-57d5be0f9944',
        owner="ETHEREUM:0x4765273c477c2dc484da4f1984639e943adccfeb",
        blockchains=["ETHEREUM", "POLYGON"]
    )
    print(f"Total NFTs owned: {nft_count}")


HTTPError: 400 Client Error: Bad Request for url: https://api.rarible.org/v0.1/balances/ETHEREUM:0x0000000000000000000000000000000000000000/ETHEREUM:0x4765273c477c2dc484da4f1984639e943adccfeb

In [17]:
import requests
from typing import List, Dict

def get_all_currencies(api_key: str) -> List[Dict]:
    """
    Fetches all supported currencies from the Rarible API.

    Args:
        api_key: Your Rarible API key.

    Returns:
        A list of dictionaries, each representing a currency with fields like:
        - `blockchain`
        - `contract`
        - `type`
        - `decimals`
        - `symbol`
        - `name`
    """
    endpoint = "https://api.rarible.org/v0.1/currencies/all"
    headers = {
        "X-API-KEY": api_key,
        "Accept": "application/json"
    }

    response = requests.get(endpoint, headers=headers)
    response.raise_for_status()
    return response.json()


# Example usage:
if __name__ == "__main__":
    API_KEY = "7162ca80-e2bc-424a-9e3d-57d5be0f9944"
    currencies = get_all_currencies(API_KEY).get('currencies')
    print(f"Total currencies found: {len(currencies)}")
    for currency in currencies:  # show first 5 for preview
        print(currency)


Total currencies found: 189
{'currencyId': 'ETHEREUM:0x4d224452801aced8b2f0aebe155379bb5d594381', 'symbol': 'apecoin', 'abbreviation': 'ape', 'rate': '0.42348946753520805', 'decimals': 18}
{'currencyId': 'IMMUTABLEX:0x4d224452801aced8b2f0aebe155379bb5d594381', 'symbol': 'apecoin', 'abbreviation': 'ape', 'rate': '0.42348946753520805', 'decimals': 18}
{'currencyId': 'ARBITRUM:0x912ce59144191c1204e64559fe8253a0e49e6548', 'symbol': 'arbitrum', 'abbreviation': 'arb', 'rate': '0.2942409077985851', 'decimals': 18}
{'currencyId': 'ETHEREUM:0xa9248f8e40d4b9c3ca8ebd8e07e9bcb942c616d8', 'symbol': 'arke', 'abbreviation': 'arke', 'rate': '0.0171081852208944', 'decimals': 4}
{'currencyId': 'IMMUTABLEX:0xa9248f8e40d4b9c3ca8ebd8e07e9bcb942c616d8', 'symbol': 'arke', 'abbreviation': 'arke', 'rate': '0.0171081852208944', 'decimals': 4}
{'currencyId': 'ETHEREUM:0x64d91f12ece7362f91a6f8e7940cd55f05060b92', 'symbol': 'ash', 'abbreviation': 'ash', 'rate': '0.6818259363749933', 'decimals': 18}
{'currencyId': 

In [16]:
import requests

API_KEY = "7162ca80-e2bc-424a-9e3d-57d5be0f9944"

response = requests.get(
    "https://api.rarible.org/v0.1/currencies/all",
    headers={"X-API-KEY": API_KEY}
)

print("Status Code:", response.status_code)
print("Response:", response.text)


Status Code: 200
Response: {"currencies":[{"currencyId":"ETHEREUM:0x4d224452801aced8b2f0aebe155379bb5d594381","symbol":"apecoin","abbreviation":"ape","rate":"0.42348946753520805","decimals":18},{"currencyId":"IMMUTABLEX:0x4d224452801aced8b2f0aebe155379bb5d594381","symbol":"apecoin","abbreviation":"ape","rate":"0.42348946753520805","decimals":18},{"currencyId":"ARBITRUM:0x912ce59144191c1204e64559fe8253a0e49e6548","symbol":"arbitrum","abbreviation":"arb","rate":"0.2942409077985851","decimals":18},{"currencyId":"ETHEREUM:0xa9248f8e40d4b9c3ca8ebd8e07e9bcb942c616d8","symbol":"arke","abbreviation":"arke","rate":"0.0171081852208944","decimals":4},{"currencyId":"IMMUTABLEX:0xa9248f8e40d4b9c3ca8ebd8e07e9bcb942c616d8","symbol":"arke","abbreviation":"arke","rate":"0.0171081852208944","decimals":4},{"currencyId":"ETHEREUM:0x64d91f12ece7362f91a6f8e7940cd55f05060b92","symbol":"ash","abbreviation":"ash","rate":"0.6818259363749933","decimals":18},{"currencyId":"IMMUTABLEX:0x64d91f12ece7362f91a6f8e7940